The color deconvolution of the images in this project was done using custom color vectors, which were produced manually using selected "optimized" images.
We used Qupath to generate those color vectors (shown in the )

In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import sys
import cv2
import os
import numpy as np
import seaborn as sns
from scipy import stats
from skimage import filters, morphology, measure, color, io
from skimage.color import separate_stains
from scipy import stats, ndimage
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import gc

import warnings
warnings.filterwarnings('ignore')


In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
#Import function
from src.functions import QuPathStainVectors

#Initialize the Class
qp = QuPathStainVectors()

# Load the averaged vectors
qp.load_vectors('../data/qupath_averaged_vectors.txt')

#BATCH PROCESSING LOOP 
def run_batch_pipeline(qp,input_base, output_base):
    for region in ['WM', 'GM']:
        for category in ['Controls', 'Patients']:
            path = os.path.join(input_base, region, category)
            if not os.path.exists(path): continue
            
            subjects = sorted([
                s for s in os.listdir(path) 
                if os.path.isdir(os.path.join(path, s)) 
                and s.startswith(('A', 'PDC'))
            ])
            
            for subject in subjects:
                sub_path = os.path.join(path, subject)
 
                # Define output paths
                h_out = os.path.join(output_base, region, category, subject, 'Hematoxylin_Cellpose')
                dab_out = os.path.join(output_base, region, category, subject, 'DAB_RGB_Verification')
                os.makedirs(h_out, exist_ok=True)
                os.makedirs(dab_out, exist_ok=True)

                print(f"Processing {subject}...")

                for tile in os.listdir(sub_path):
                    if tile.lower().endswith(('.jpg', '.png')):
                        img_rgb = cv2.cvtColor(cv2.imread(os.path.join(sub_path, tile)), cv2.COLOR_BGR2RGB)
                            
                        #Perform Math Separation
                        h_od, dab_od = qp.deconvolve_image(img_rgb)
                            
                        #Save Hematoxylin as Grayscale
                        h_gray = np.clip(h_od * 255, 0, 255).astype(np.uint8)
                        cv2.imwrite(os.path.join(h_out, tile), h_gray)

                        #Save DAB as RGB Reconstruction (Best for visualization)
                        # We use the DAB vector to "re-stain" the isolated signal
                        dab_rgb_vis = 255 * np.exp(-dab_od[:, :, np.newaxis] * qp.dab_vector)
                        dab_rgb_vis = np.clip(dab_rgb_vis, 0, 255).astype(np.uint8)
                            
                        # Convert to BGR for OpenCV saving
                        dab_bgr_vis = cv2.cvtColor(dab_rgb_vis, cv2.COLOR_RGB2BGR)
                        cv2.imwrite(os.path.join(dab_out, tile), dab_bgr_vis)

                gc.collect()

In [ ]:
#EXECUTION (adjust the input_base and the output_base as needed)
run_batch_pipeline(qp, r'input_base', r'output_base')